# MyTravelHelper: Streamlit + Hugging Face Inference Providers

Tài liệu Jupyter Notebook này trình bày quy trình xây dựng ứng dụng trợ lý du lịch **MyTravelHelper** theo đúng các yêu cầu và tác vụ nâng cao trong kế hoạch dự án.

## 1. Mục tiêu và phân tích yêu cầu

### 1.1 Mục tiêu
Xây dựng ứng dụng **MyTravelHelper** — một hệ thống NLP hỗ trợ du lịch, có khả năng:
- Phân tích cảm xúc (bao gồm phân tích theo khía cạnh - ABSA) đối với review du lịch.
- Phân loại ý định (Intent) và trích xuất thực thể (NER) từ yêu cầu của người dùng.
- Phát hiện và gán nhãn chủ đề tự động trong tập review du lịch (Topic Modeling).

### 1.2 Phân tích yêu cầu
- **Functional Requirements:**
  - FR-01: Phân tích cảm xúc tổng thể (pos/neg/neutral)
  - FR-02: Phân tích cảm xúc theo khía cạnh (ABSA)
  - FR-03: Phân loại ý định người dùng (booking, info, complaint...)
  - FR-04: Trích xuất thực thể: địa điểm, thời gian, số lượng
  - FR-05: Phát hiện chủ đề trong tập review du lịch
  - FR-06: Giao diện Streamlit đơn giản, trực quan
  - FR-07: Hiển thị confidence score cho từng kết quả
- **Non-Functional Requirements:**
  - Latency: Phản hồi trong vòng ≤ 5 giây / request
  - Reliability: Graceful fallback khi offline hoặc lỗi API
  - Reproducibility: Chạy lại notebook từ đầu không lỗi

## 2. Kiến trúc tổng quát

Hệ thống được thiết kế theo luồng xử lý tuần tự (Sequential Pipeline) tách biệt giao diện và xử lý logic:

```mermaid
flowchart TD
    A[User Input] --> B[Streamlit UI]
    B --> C{Task Type?}
    C -->|Review text| D[Sentiment Module]
    C -->|Travel query| E[Intent + NER Module]
    C -->|Batch reviews| F[Topic Module]
    D --> G[HF Inference API\ndeberta-v3-absa]
    E --> H[HF Inference API\nbart-large-mnli + bert-NER]
    F --> I[BERTopic\n+ sentence-transformers]
    G --> J[Output Aggregator]
    H --> J
    I --> J
    J --> K[Streamlit Result View\nCharts · Badges · Entities]
```

## 3. Thiết lập Hugging Face Inference

### 3.1 Nạp các thư viện cần thiết
Chúng ta nạp các thư viện chính cho bài toán.

In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
from modules import get_hf_token, get_inference_client

print("Các thư viện được nạp thành công!")

Các thư viện được nạp thành công!


### 3.2 Khởi tạo client & Health Check
Chúng ta đọc Hugging Face Token từ file `.env` và kiểm tra kết nối với API.

In [2]:
token = get_hf_token()
client = get_inference_client()

if token:
    print("Tìm thấy HF_TOKEN: hf_..." + token[-6:])
else:
    print("Không tìm thấy HF_TOKEN. Hệ thống sẽ sử dụng chế độ Heuristic Fallback.")

Không tìm thấy HF_TOKEN. Hệ thống sẽ sử dụng chế độ Heuristic Fallback.


## 4. Giới thiệu & So sánh Model

Dưới đây là bảng so sánh lựa chọn mô hình cho các tác vụ của hệ thống:

### 4.1 Sentiment Analysis & ABSA
| Model | Ưu điểm | Nhược điểm | Kết luận |
|---|---|---|---|
| `sst2` | Siêu nhẹ, nhanh | Chỉ có Pos/Neg, không hỗ trợ Aspect | Baseline |
| `twitter-roberta-base-sentiment-latest` | Hỗ trợ 3 nhãn (Pos/Neg/Neu), dữ liệu thực tế | Không có Aspect | Chọn cho Basic |
| `deberta-v3-base-absa-v1.1` | Đúng tác vụ, phân tích theo aspect | Cần truyền aspect prompt | **Chọn cho Advanced ABSA** |

### 4.2 Intent Classification & NER
| Model | Vai trò | Mô tả |
|---|---|---|
| `facebook/bart-large-mnli` | Intent classification | Zero-shot classification linh hoạt |
| `dslim/bert-base-NER` | Named Entity Recognition | Trích xuất các thực thể chuẩn (LOC, PER, ORG) |

### 4.3 Topic Modeling
| Model | Ưu điểm | Nhược điểm | Kết luận |
|---|---|---|---|
| LDA | Nhẹ, không cần GPU | Chất lượng kém trên văn bản ngắn | Không dùng |
| `BERTopic` | Gom cụm chất lượng cao dựa trên Embeddings | Cần nhiều RAM / GPU | **Chọn cho Advanced Topic** |

## 5. Test cơ bản thiết lập môi trường
Chúng ta thực hiện gọi các API cơ bản để đảm bảo kết nối ổn định.

In [3]:
from modules.sentiment import analyze_sentiment

sample_review = "The room was spotless and very clean, but the staff was rude."
basic_res = analyze_sentiment(sample_review, mode="basic")
print("Basic Sentiment:", basic_res)

Basic Sentiment: {'label': 'POSITIVE', 'score': 0.8, 'source': 'heuristic'}


## 6. Chạy ứng dụng & Kiểm thử các chức năng
Ứng dụng chính của chúng ta chạy bằng Streamlit:
```bash
streamlit run app.py
```

Chúng ta sẽ kiểm thử các API logic trực tiếp ngay tại đây để chứng minh tính đúng đắn.

In [4]:
from modules.intent_ner import classify_intent, extract_entities

query = "Tôi muốn đặt khách sạn ở Đà Nẵng 3 đêm với chi phí 5 triệu đồng."
intent = classify_intent(query)
entities = extract_entities(query)

print("Intent:", intent)
print("Entities:", entities)

Intent: {'intent': 'book hotel', 'confidence': 0.65, 'source': 'heuristic'}
Entities: [{'word': 'Đà Nẵng', 'entity_type': 'LOCATION', 'score': 1.0, 'start': 25, 'end': 32, 'source': 'rules'}, {'word': '3 đêm', 'entity_type': 'DURATION', 'score': 0.95, 'start': 33, 'end': 38, 'source': 'rules'}, {'word': '5 triệu', 'entity_type': 'BUDGET', 'score': 0.95, 'start': 51, 'end': 58, 'source': 'rules'}]


## 7. Phần nâng cao (Advanced NLP)

### 7.1 Phân tích cảm xúc theo khía cạnh (ABSA)
Sử dụng mô hình `yangheng/deberta-v3-base-absa-v1.1` để phân tích cảm xúc chi tiết trên từng thuộc tính.

In [5]:
absa_res = analyze_sentiment(
    "Khách sạn có vị trí ngay trung tâm, rất gần biển nên đi bộ ra ngoài rất tiện. Tuy nhiên phòng hơi nhỏ và ẩm mốc.",
    mode="absa"
)
print("ABSA Result:")
for r in absa_res:
    print(f"- Aspect: {r['aspect']} -> Sentiment: {r['sentiment']} (Confidence: {r['confidence']})")

ABSA Result:
- Aspect: vị trí -> Sentiment: NEUTRAL (Confidence: 0.5)
- Aspect: tiện nghi -> Sentiment: NEUTRAL (Confidence: 0.5)


### 7.2 Phân loại ý định & NER kết hợp
Luồng xử lý chatbot tích hợp trả lời câu hỏi và phân loại thông tin.

In [6]:
from modules.intent_ner import travel_chat

test_chat = "Gợi ý lịch trình đi Huế 2 ngày từ Đà Nẵng"
response = travel_chat(test_chat)
print("Chatbot Response:")
print(response)

Chatbot Response:
Gợi ý nhanh: Hãy xác định rõ điểm đến, thời gian chuyến đi, ngân sách dự kiến và phong cách du lịch của bạn. Với câu hỏi 'Gợi ý lịch trình đi Huế 2 ngày từ Đà Nẵng', bạn có thể tham khảo lịch trình cơ bản 2 ngày 1 đêm, lựa chọn các địa điểm gần nhau để thuận tiện di chuyển, kết hợp thử các món ăn đặc sản địa phương.


### 7.3 Phát hiện chủ đề (BERTopic Topic Modeling)
Gom cụm các review du lịch và dán nhãn chủ đề tự động.

In [7]:
import pandas as pd
from modules.topic import detect_topics_bertopic

reviews = [
    "Khách sạn có bể bơi vô cực siêu đẹp, rất thích hợp nghỉ dưỡng.",
    "Đồ ăn sáng ở nhà hàng buffet rất phong phú và hải sản tươi ngon.",
    "Phòng ốc hơi nhỏ nhưng dọn dẹp rất sạch sẽ hàng ngày.",
    "Giá phòng quá đắt so với dịch vụ nghèo nàn ở đây.",
    "Vị trí gần sân bay dễ di chuyển bằng taxi."
]

topics, model = detect_topics_bertopic(reviews)
info = model.get_topic_info()
print("Topic Info DataFrame:")
print(info)

Topic Info DataFrame:
   Topic  Count                          Name            Representation
0      0      3          0_Ẩm thực địa phương  [rất, hàng, ốc, sẽ, dẹp]
1      1      2  1_Dịch vụ khách sạn & Lễ tân  [nàn, đắt, đây, vụ, với]


## 8. Hình ảnh Minh họa Giao diện Ứng dụng Streamlit

Phần này chứa các hình ảnh chụp màn hình giao diện thực tế của ứng dụng **MyTravelHelper** để giáo viên có thể dễ dàng đánh giá trực quan mà không cần chạy trực tiếp server Streamlit.

> **Lưu ý:** Bạn hãy chụp ảnh màn hình các tab tương ứng của Streamlit app, đổi tên và lưu vào thư mục `assets/` theo đúng tên file tương ứng để hiển thị trong notebook.

### 8.1 Giao diện Tab 1: Chatbot Tư Vấn & Phân Tích NLU
Hiển thị khung trò chuyện, phân loại ý định người dùng (Intent) và trích xuất thực thể nổi bật (Entities: Location, Date, Duration, Budget) với màu sắc bắt mắt.

![Tab 1: Chatbot & NLU](assets/screenshot_chatbot.jpg)

### 8.2 Giao diện Tab 2: Phân Tích Cảm Xúc Khía Cạnh (ABSA)
Hiển thị chỉ số cảm xúc chung và lưới thẻ màu trực quan đại diện cho từng khía cạnh khách sạn (Vị trí, Dịch vụ, Phòng, v.v.) kèm điểm tin cậy.

![Tab 2: Phân tích ABSA](assets/screenshot_sentiment.jpg)

### 8.3 Giao diện Tab 3: Gom Cụm Chủ Đề Review (Topic Modeling)
Hiển thị biểu đồ cột phân bố các nhóm chủ đề được phát hiện trong tập đánh giá, cùng với từ khóa đại diện và số lượng tương ứng.

![Tab 3: Gom cụm chủ đề](assets/screenshot_topics_1.jpg)
![Tab 3: Phân loại chủ đề](assets/screenshot_topics_2.jpg)

### 8.4 Giao diện Tab 4: Sơ Đồ Kiến Trúc Hệ Thống & Trạng Thái
Hiển thị sơ đồ Pipeline Mermaid cùng với bảng chẩn đoán kết nối API / trạng thái môi trường hiện tại của trợ lý.

![Tab 4: Sơ đồ kiến trúc](assets/screenshot_pipeline.jpg)